# 00 Bootstrap harness: Spreekorb capstone
Save as notebooks/00_bootstrap_harness.ipynb in your repo and run top to bottom in VS Code (it finds the repo root itself). Creates folders, agent harness and repo basics.
- Existing files are backed up to `_backup/<timestamp>/` before they are overwritten.
- Each step prints PASS or FAIL.
- Undo (cell 7) only runs if you set `RUN_UNDO = True`.
- Git commands are printed, never run.

In [1]:
# 0 Settings
from pathlib import Path
import json, shutil, subprocess, sys, datetime

# repo root = git top folder, so the notebook can sit in notebooks/
_top = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd()
DRY_RUN = False              # True = only show what would be created
RUN_UNDO = False             # True = cell 7 undoes this bootstrap
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP = REPO / "_backup" / STAMP
results = []                 # (step, PASS/FAIL, note)

def check(step, ok, note=""):
    results.append((step, "PASS" if ok else "FAIL", note))
    print(("PASS" if ok else "FAIL"), "|", step, "|", note)

print("Repo:", REPO)

Repo: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption


In [2]:
# 1 Check folder and git
def git(*args):
    r = subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True)
    return r.returncode, r.stdout.strip()

code_, _ = git("rev-parse", "--is-inside-work-tree")
check("git repo", code_ == 0, str(REPO))
_, branch = git("branch", "--show-current")
check("branch", True, branch or "(no commits yet)")
_, tracked_env = git("ls-files", ".env")
check(".env not tracked by git", tracked_env == "", tracked_env or "ok")
check("repo root is not a subfolder", REPO.name not in [".claude", "notebooks"], REPO.name)
assert all(r[1] == "PASS" for r in results), "Stop: fix the FAIL above before continuing"

PASS | git repo | /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption
PASS | branch | main
PASS | .env not tracked by git | ok
PASS | repo root is not a subfolder | ecommerce_customersatisfaction_adiadoption


In [3]:
# 2 Folders to create
FOLDERS = [
    "data/raw", "data/processed", "research", "dashboard", "n8n", "langsmith",
    "cost_estimation", "feedback", "notebooks", "specs", "decisions", "logs",
    ".claude/agents", ".claude/hooks",
]
for f in FOLDERS:
    p = REPO / f
    if not DRY_RUN:
        p.mkdir(parents=True, exist_ok=True)
    check("folder " + f, DRY_RUN or p.is_dir())

PASS | folder data/raw | 
PASS | folder data/processed | 
PASS | folder research | 
PASS | folder dashboard | 
PASS | folder n8n | 
PASS | folder langsmith | 
PASS | folder cost_estimation | 
PASS | folder feedback | 
PASS | folder notebooks | 
PASS | folder specs | 
PASS | folder decisions | 
PASS | folder logs | 
PASS | folder .claude/agents | 
PASS | folder .claude/hooks | 


In [4]:
# 3 Harness files (agents, hooks, board, specs, logs)
FILES = {}

FILES["CLAUDE.md"] = """# Claude Code entry point
@AGENTS.md
"""

FILES["AGENTS.md"] = """# AGENTS.md: Spreekorb capstone

## Project
Ironhack AI Consulting capstone. Client: Spreekorb, invented Berlin e-commerce company, about 500 employees, no AI yet.
CEO Chleo fears AI is a black box. Focus: email customer service, SLA (first response time) and CSAT.
Use cases: UC1 email triage + reply drafts (human approves), UC2 root cause report to the business, UC3 team-level view.

## Stack
Python 3.12, Jupyter in VS Code, pandas, OpenAI gpt-4o-mini, LangSmith, n8n, Tableau (.twbx + Tableau Public).

## Repo map
data/raw (downloads, not committed) | data/processed | research | dashboard | n8n | langsmith | cost_estimation | feedback | notebooks | specs | decisions | logs

## Card flow
board.md is the communication board. Status: To do > In progress > Review > Approved > Done.
1. Ayhan names the card. Read its spec in specs/ first.
2. builder writes ONE task notebook: notebooks/<card-id>_<short-name>.ipynb.
3. reviewer checks the notebook against the spec. compliance-checker and data-checker when the spec says so.
4. Ayhan runs the notebook, then moves the card to Approved. Only Ayhan sets Approved or Done.
5. Write a one-line handoff note on the card in board.md.

## Task notebook pattern (every card)
Cells, short and beginner-friendly, # comments:
1. Check: right folder, input files exist. Print PASS/FAIL.
2. Backup: copy every file you will change to _backup/<timestamp>/.
3. Create: do the card's steps (write files with %%writefile or Python).
4. Verify: each acceptance item from the spec prints PASS/FAIL.
5. Unchanged: prove existing files you did not own are unchanged.
6. Undo: restore the backup (runs only if RUN_UNDO = True).
7. Git: print git commands, never run them.

## Definition of Done
All spec checks PASS, reviewer OK, Ayhan approved, board.md updated.

## Never
- Never commit, push or reset git. Ayhan commits.
- Never read or write .env or real keys. Use .env.example names only.
- Never use real personal data. Never record agent names or IDs; team level only, minimum group of 5.
- Never invent numbers or sources. Cite, or label "estimate" or "synthetic".
- Never touch files outside the card's spec. Smallest change that meets the spec.
"""

FILES[".claude/settings.json"] = json.dumps({
    "permissions": {"deny": ["Read(./.env)", "Edit(./.env)", "Write(./.env)"]},
    "hooks": {
        "PreToolUse": [{"matcher": "Bash|Edit|Write|NotebookEdit", "hooks": [
            {"type": "command", "command": "python3 \"$CLAUDE_PROJECT_DIR/.claude/hooks/guard.py\""}]}],
        "PostToolUse": [{"matcher": "Bash|Edit|Write|NotebookEdit", "hooks": [
            {"type": "command", "command": "python3 \"$CLAUDE_PROJECT_DIR/.claude/hooks/log_action.py\""}]}],
    },
}, indent=2) + "\n"

FILES[".claude/hooks/guard.py"] = """# Blocks git writes and .env access by agents (exit 2 = blocked)
import json, sys

data = json.load(sys.stdin)
tool = data.get("tool_name", "")
inp = data.get("tool_input", {})
cmd = inp.get("command", "")
path = inp.get("file_path", "") or inp.get("notebook_path", "")

blocked_cmds = ["git commit", "git push", "git reset", "git merge", "rm -rf"]
if tool == "Bash" and any(b in cmd for b in blocked_cmds):
    print("Blocked: git writes and rm -rf are for Ayhan only.", file=sys.stderr)
    sys.exit(2)
if tool == "Bash" and ".env" in cmd and ".env.example" not in cmd:
    print("Blocked: agents never touch .env.", file=sys.stderr)
    sys.exit(2)
if path.endswith(".env"):
    print("Blocked: agents never write .env.", file=sys.stderr)
    sys.exit(2)
sys.exit(0)
"""

FILES[".claude/hooks/log_action.py"] = """# Appends one line per agent action to logs/agent_log.md
import json, sys, os, datetime

data = json.load(sys.stdin)
inp = data.get("tool_input", {})
target = inp.get("file_path") or inp.get("notebook_path") or inp.get("command", "")
line = "| {} | {} | {} |\\n".format(
    datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    data.get("tool_name", ""),
    str(target).replace("|", "/")[:120],
)
root = os.environ.get("CLAUDE_PROJECT_DIR", ".")
with open(os.path.join(root, "logs", "agent_log.md"), "a") as f:
    f.write(line)
"""

FILES[".claude/agents/builder.md"] = """---
name: builder
description: Builds the task notebook for the card Ayhan names. Use for any build card.
tools: Read, Write, Edit, Glob, Grep
model: sonnet
---
Read AGENTS.md and the card's spec in specs/. Write one task notebook in notebooks/ following the task notebook pattern.
Keep code short, beginner-friendly, # comments. Only touch files the spec names. Do not run git. End with a 3-line report: what changed, checks, risks.
"""

FILES[".claude/agents/reviewer.md"] = """---
name: reviewer
description: Reviews a finished task notebook against its spec. Use after builder, before Ayhan runs it.
tools: Read, Glob, Grep
model: sonnet
---
Judge only the output and the spec, not how it was built. For each acceptance item say PASS or FAIL with one reason.
Flag files changed outside the spec, missing undo or backup, invented numbers. Max 10 lines.
"""

FILES[".claude/agents/compliance-checker.md"] = """---
name: compliance-checker
description: Checks GDPR, EU AI Act and works council risks in a card's output. Use when the spec says compliance check.
tools: Read, Glob, Grep
model: sonnet
---
Check: no real personal data, no agent names or IDs, team level only (min group 5), synthetic data labelled,
human approves every AI output, no overclaiming ("prepared", not "fully compliant"). List issues only. Max 10 lines.
"""

FILES[".claude/agents/data-checker.md"] = """---
name: data-checker
description: Checks data sources, numbers and test sets in a card's output. Use when the spec says data check.
tools: Read, Glob, Grep
model: sonnet
---
Check: every number has a source or is labelled estimate/synthetic, dataset links present,
test cases kept apart from tuning cases, no AI grading itself. List issues only. Max 10 lines.
"""

FILES["board.md"] = """# Board: Spreekorb capstone (Round 1, submit Mon 30 Nov 2026 before 18:00 as project-5)

Status: To do > In progress > Review > Approved > Done. Only Ayhan sets Approved and Done.

| ID | Card | Spec | Status | Evidence | Handoff note |
|---|---|---|---|---|---|
| R1-00 | Bootstrap harness | (this notebook) | In progress | notebooks/00_bootstrap_harness.ipynb | |
| R1-01 | Research pack | specs/R1-01_research.md | To do | | |
| R1-02 | Data prep | specs/R1-02_data.md | To do | | |
| R1-03 | Tableau dashboard | specs/R1-03_dashboard.md | To do | | |
| R1-04 | n8n POC | specs/R1-04_n8n_poc.md | To do | | |
| R1-05 | LangSmith sample | specs/R1-05_langsmith.md | To do | | |
| R1-06 | Timing + cost + timeline | specs/R1-06_cost_timeline.md | To do | | |
| R1-07 | README + submit | specs/R1-07_submit.md | To do | | |
| R1-08 | Round 1 decision | specs/R1-08_decision.md | To do | | |
"""

SPECS = {
 "R1-01_research": ("Research pack (rubric: 20 pts with use cases)",
   "research/sector_research.md, research/opportunities_risks.md, research/use_cases.md",
   ["Sector, size (500 = large), why size matters", "Problems with numbers, each cited or marked estimate",
    "Data described in 2-3 sentences: what, where from, why it fits", "Opportunity/risk table + one risk avoided on purpose",
    "2-3 use cases with why they fit a large company"], "data check"),
 "R1-02_data": ("Data prep",
   "data/processed/ files + data/README.md",
   ["Shopzilla filtered to email channel, agent/supervisor/manager names dropped, teams anonymised",
    "Tobi-Bueck filtered to e-commerce queues", "Synthetic team pulse (Yes/No + why category) labelled synthetic, team level only",
    "Held-out test set kept apart from tuning set", "Raw files not committed, links in data/README.md"], "data check, compliance check"),
 "R1-03_dashboard": ("Tableau dashboard (rubric: 25 pts)",
   "dashboard/dashboard.twbx, dashboard/dashboard_documentation.md",
   ["5-7 metrics, each tagged A problem / B use case / C trust", "Headlines are insights, not chart labels",
    "Sources on the dashboard; synthetic tiles labelled", "Tableau Public link + screenshots in docs",
    "Test: Chleo understands the problem and why to solve it in 2 minutes"], "data check"),
 "R1-04_n8n_poc": ("n8n POC for UC1 (rubric: 25 pts with LangSmith)",
   "n8n/workflow.json, n8n/workflow_documentation.md",
   ["3-4 nodes: trigger, AI triage + draft, output for human review", "Runs on sample emails",
    "Docs: what each node does, why it fits, limits vs production", "No email is sent automatically"], "compliance check"),
 "R1-05_langsmith": ("LangSmith monitoring sample",
   "langsmith/ (test items, setup notes, screenshots)",
   ["10 items traced: input, output, time, cost", "Short note: what it shows about transparency",
    "Real cost and time per draft handed to R1-06"], "data check"),
 "R1-06_cost_timeline": ("Timing + cost + timeline (rubric: 15 pts with docs)",
   "data/timing_log.csv, cost_estimation/cost_analysis.md, cost_estimation/timeline_estimate.md",
   ["Timing: 10 manual vs 10 AI-assisted replies, held-out emails", "4 costs: build (day rate x days), AI calls, tools, value",
    "Assumptions table with sources", "Timeline: prototype > pilot > rollout"], "data check"),
 "R1-07_submit": ("README + submission",
   "README.md, requirements.txt, .env.example",
   ["README: overview, what is where, how to view dashboard and run POC", "No secrets in repo",
    "All R1 checklist items present", "Submitted on portal as project-5 before Mon 30 Nov 18:00"], "reviewer"),
 "R1-08_decision": ("Round 1 decision",
   "feedback/round1_decision.md",
   ["Feedback summary 3-5 bullets (or: no feedback received)", "KEEP or CHANGE + why",
    "Round 2 focus: POC improvements, MVP scope (one capability), compliance/ROI priorities"], "reviewer"),
}
for name, (title, outputs, accept, checks) in SPECS.items():
    FILES["specs/" + name + ".md"] = (
        "# " + name.split("_")[0] + " " + title + "\n\n"
        + "**Outputs:** " + outputs + "\n\n**Acceptance (each prints PASS/FAIL):**\n"
        + "".join("- " + a + "\n" for a in accept)
        + "\n**Checks:** reviewer, " + checks + "\n\n**Out of scope:** anything not listed above.\n")

FILES["decisions/decision_log.md"] = """# Decision log

| # | Date | Decision |
|---|---|---|
| D0 | 26 Nov 2026 | Spreekorb, Berlin e-commerce, ~500 people, no AI. Email channel: SLA (first response time) + CSAT. UC1 triage + drafts (human approves), UC2 root cause report, UC3 team view (no agent data). Data: Shopzilla (real), Tobi-Bueck (public), synthetic team pulse (labelled). AHT dropped; time saved measured on 10 held-out emails. |
| D1 | 26 Nov 2026 | Harness: Claude Code, subagents builder/reviewer/compliance-checker/data-checker, hooks guard + log, board.md, specs/, one task notebook per card. Work on main; agents never commit. |
"""

FILES["logs/agent_log.md"] = """# Agent log (written by .claude/hooks/log_action.py)

| Time | Tool | Target |
|---|---|---|
"""

def write(rel, text):
    p = REPO / rel
    if DRY_RUN:
        print("would write", rel); return
    if p.exists() and p.read_text() != text:
        dest = BACKUP / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
    created_now = not p.exists()
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
    if created_now:
        CREATED.append(rel)

CREATED = []
for rel, text in FILES.items():
    write(rel, text)
    check("file " + rel, DRY_RUN or (REPO / rel).exists())

PASS | file CLAUDE.md | 
PASS | file AGENTS.md | 
PASS | file .claude/settings.json | 
PASS | file .claude/hooks/guard.py | 
PASS | file .claude/hooks/log_action.py | 
PASS | file .claude/agents/builder.md | 
PASS | file .claude/agents/reviewer.md | 
PASS | file .claude/agents/compliance-checker.md | 
PASS | file .claude/agents/data-checker.md | 
PASS | file board.md | 
PASS | file specs/R1-01_research.md | 
PASS | file specs/R1-02_data.md | 
PASS | file specs/R1-03_dashboard.md | 
PASS | file specs/R1-04_n8n_poc.md | 
PASS | file specs/R1-05_langsmith.md | 
PASS | file specs/R1-06_cost_timeline.md | 
PASS | file specs/R1-07_submit.md | 
PASS | file specs/R1-08_decision.md | 
PASS | file decisions/decision_log.md | 
PASS | file logs/agent_log.md | 


In [5]:
# 4 Repo basics
BASICS = {}
BASICS["README.md"] = """# Spreekorb: AI for email customer service (Ironhack capstone)

Independent case study. Spreekorb is an invented company. Public and labelled synthetic data only.

## What is where
| Folder | Content |
|---|---|
| research/ | Sector research, opportunities and risks, use cases |
| data/ | Processed data and data README (raw downloads not committed) |
| dashboard/ | Tableau workbook (.twbx) and documentation with Tableau Public link |
| n8n/ | POC workflow export and documentation |
| langsmith/ | Monitoring sample: test items, setup, screenshots |
| cost_estimation/ | Cost analysis and timeline |
| feedback/ | Round 1 decision |
| notebooks/ | One task notebook per card (build evidence) |
| specs/, board.md, decisions/, logs/ | How the project was built: specs, board, decisions, agent log |

## Setup
1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and add your keys.
3. Open the notebooks in VS Code and run top to bottom.
"""
BASICS["requirements.txt"] = "pandas\njupyter\npython-dotenv\nopenai\nlangsmith\n"
BASICS[".env.example"] = """OPENAI_API_KEY=
LANGSMITH_API_KEY=
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=spreekorb-capstone
"""
BASICS[".gitignore"] = """.env
_backup/
data/raw/
.ipynb_checkpoints/
__pycache__/
.DS_Store
"""
BASICS["data/timing_log.csv"] = "email_id,mode,minutes,notes\n"

for rel, text in BASICS.items():
    if (REPO / rel).exists() and rel in ["README.md", ".gitignore"]:
        print("exists, backed up and replaced:", rel)
    write(rel, text)
    check("file " + rel, DRY_RUN or (REPO / rel).exists())

exists, backed up and replaced: README.md
PASS | file README.md | 
PASS | file requirements.txt | 
PASS | file .env.example | 
exists, backed up and replaced: .gitignore
PASS | file .gitignore | 
PASS | file data/timing_log.csv | 


In [6]:
# 5 Verify
if not DRY_RUN:
    # settings.json is valid JSON
    try:
        json.loads((REPO / ".claude/settings.json").read_text()); ok = True
    except Exception as e:
        ok = False
    check("settings.json valid JSON", ok)

    # guard hook blocks git commit and .env, allows normal commands
    guard = REPO / ".claude/hooks/guard.py"
    def run_guard(payload):
        r = subprocess.run([sys.executable, str(guard)], input=json.dumps(payload), capture_output=True, text=True)
        return r.returncode
    check("guard blocks git commit", run_guard({"tool_name": "Bash", "tool_input": {"command": "git commit -m x"}}) == 2)
    check("guard blocks writing .env", run_guard({"tool_name": "Write", "tool_input": {"file_path": str(REPO / ".env")}}) == 2)
    check("guard allows ls", run_guard({"tool_name": "Bash", "tool_input": {"command": "ls"}}) == 0)
    check("guard allows .env.example", run_guard({"tool_name": "Write", "tool_input": {"file_path": str(REPO / ".env.example")}}) == 0)

    # subagents have name + description
    for a in (REPO / ".claude/agents").glob("*.md"):
        t = a.read_text()
        check("agent " + a.name, "name:" in t and "description:" in t)

    # .env stays ignored
    check(".env in .gitignore", ".env" in (REPO / ".gitignore").read_text().split())

fails = [r for r in results if r[1] == "FAIL"]
print("\nSUMMARY:", len(results) - len(fails), "PASS,", len(fails), "FAIL")
for f in fails: print("  FAIL:", f[0], f[2])

PASS | settings.json valid JSON | 
PASS | guard blocks git commit | 
PASS | guard blocks writing .env | 
PASS | guard allows ls | 
PASS | guard allows .env.example | 
PASS | agent reviewer.md | 
PASS | agent builder.md | 
PASS | agent data-checker.md | 
PASS | agent compliance-checker.md | 
PASS | .env in .gitignore | 

SUMMARY: 53 PASS, 0 FAIL


In [7]:
# 6 Save manifest (used by undo)
if not DRY_RUN:
    BACKUP.mkdir(parents=True, exist_ok=True)
    (BACKUP / "manifest.json").write_text(json.dumps({"created": CREATED}, indent=2))
    print("Manifest:", BACKUP / "manifest.json", "|", len(CREATED), "new files")

Manifest: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261126_152411/manifest.json | 23 new files


In [ ]:
# 7 Undo (only if RUN_UNDO = True): delete new files, restore backups
if RUN_UNDO:
    last = sorted((REPO / "_backup").iterdir())[-1]
    created = json.loads((last / "manifest.json").read_text())["created"]
    for rel in created:
        (REPO / rel).unlink(missing_ok=True)
    for f in last.rglob("*"):
        if f.is_file() and f.name != "manifest.json":
            shutil.copy2(f, REPO / f.relative_to(last))
    check("undo", True, f"removed {len(created)} files, restored backups from {last.name}")
else:
    print("Undo skipped (RUN_UNDO = False)")

In [8]:
# 8 Git commands (copy and run yourself)
print("git add .")
print('git commit -m "R1-00 bootstrap harness: agents, hooks, board, specs"')
print("git push")
print("\nThen: restart Claude Code in VS Code so it loads .claude/agents and hooks.")

git add .
git commit -m "R1-00 bootstrap harness: agents, hooks, board, specs"
git push

Then: restart Claude Code in VS Code so it loads .claude/agents and hooks.
